# Grafo no Neo4j

Monta o grafo a partir do DW (Oracle): municípios, regiões e macrorregiões de saúde,
hospitais, especialidades, procedimentos, meses e perfis de paciente. O grafo é todo
derivado do DW, então é reconstruído do zero a cada execução, sem duplicar.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import consultar, conectar_neo4j, gravar_grafo
driver = conectar_neo4j()

## Limpeza e chaves únicas

In [ ]:
with driver.session() as s:
    s.run("MATCH (n) CALL { WITH n DETACH DELETE n } IN TRANSACTIONS OF 10000 ROWS").consume()
    for rotulo, chave in [("Macrorregiao", "nome"), ("Regiao", "nome"), ("Municipio", "cod_ibge"),
                          ("Hospital", "cnes"), ("Especialidade", "cod"), ("Procedimento", "cod"),
                          ("Mes", "ano_mes"), ("Perfil", "chave"), ("GrupoDoenca", "nome")]:
        s.run(f"CREATE CONSTRAINT {rotulo.lower()}_{chave} IF NOT EXISTS "
              f"FOR (n:{rotulo}) REQUIRE n.{chave} IS UNIQUE").consume()
print("grafo limpo")

## Municípios, regiões e macrorregiões

In [ ]:
df = consultar("""
    SELECT COD_IBGE_6 AS COD, NOME_MUNICIPIO AS NOME, UF, REGIAO_SAUDE AS REGIAO,
           MACRORREGIAO AS MACRO, TIPO_LOCALIDADE AS TIPO
    FROM DIM_MUNICIPIO
    WHERE SK_MUNICIPIO <> -1
      AND (UF = 'AL' OR SK_MUNICIPIO IN (SELECT SK_MUNICIPIO_RESIDENCIA FROM FATO_INTERNACAO))
""")
gravar_grafo(driver, """
    UNWIND $linhas AS r
    MERGE (ma:Macrorregiao {nome: r.macro})
    MERGE (rg:Regiao {nome: r.regiao})
    MERGE (rg)-[:PERTENCE_A]->(ma)
    MERGE (m:Municipio {cod_ibge: r.cod})
    SET m.nome = r.nome, m.uf = r.uf, m.tipo = r.tipo
    MERGE (m)-[:PERTENCE_A]->(rg)
""", df, "Municipio")

## Hospitais (com leitos SUS e taxa de ocupação)

In [ ]:
df = consultar("""
    WITH ocup AS (
        SELECT COD_CNES, ROUND(AVG(QTD_LEITOS_SUS)) AS LEITOS_SUS,
               ROUND(100 * SUM(QTD_PACIENTES_DIA) / NULLIF(SUM(QTD_LEITOS_DIA), 0), 1) AS PCT_OCUPACAO
        FROM MART_P04_OCUPACAO_HOSPITALAR GROUP BY COD_CNES
    )
    SELECT e.COD_CNES AS CNES, e.NOME_ESTABELECIMENTO AS NOME, e.TIPO_UNIDADE AS TIPO,
           e.TIPO_GESTAO AS GESTAO, STATS_MODE(m.COD_IBGE_6) AS COD_MUN,
           MAX(o.LEITOS_SUS) AS LEITOS_SUS, MAX(o.PCT_OCUPACAO) AS PCT_OCUPACAO
    FROM FATO_INTERNACAO f
    JOIN DIM_ESTABELECIMENTO e ON e.SK_ESTABELECIMENTO = f.SK_ESTABELECIMENTO
    JOIN DIM_MUNICIPIO m       ON m.SK_MUNICIPIO = f.SK_MUNICIPIO_ATENDIMENTO
    LEFT JOIN ocup o           ON o.COD_CNES = e.COD_CNES
    WHERE f.SK_ESTABELECIMENTO <> -1
    GROUP BY e.COD_CNES, e.NOME_ESTABELECIMENTO, e.TIPO_UNIDADE, e.TIPO_GESTAO
""")
gravar_grafo(driver, """
    UNWIND $linhas AS r
    MERGE (h:Hospital {cnes: r.cnes})
    SET h.nome = r.nome, h.tipo = r.tipo, h.gestao = r.gestao,
        h.leitos_sus = r.leitos_sus, h.pct_ocupacao = r.pct_ocupacao
    WITH h, r
    MATCH (m:Municipio {cod_ibge: r.cod_mun})
    MERGE (h)-[:LOCALIZADO_EM]->(m)
""", df, "Hospital")

## Município de residência internado em hospital

In [ ]:
df = consultar("""
    SELECT m.COD_IBGE_6 AS MUN, e.COD_CNES AS CNES,
           SUM(f.QTD_INTERNACAO) AS INTERNACOES, SUM(f.VL_TOTAL) AS VALOR,
           SUM(f.QTD_DIAS_PERMANENCIA) AS DIAS,
           SUM(CASE WHEN c.TIPO_CARATER = 'URGENCIA' THEN f.QTD_INTERNACAO ELSE 0 END) AS URGENCIAS,
           SUM(CASE WHEN d.FLG_RESPIRATORIA = 'S' THEN f.QTD_INTERNACAO ELSE 0 END) AS RESPIRATORIAS
    FROM FATO_INTERNACAO f
    JOIN DIM_MUNICIPIO m           ON m.SK_MUNICIPIO = f.SK_MUNICIPIO_RESIDENCIA
    JOIN DIM_ESTABELECIMENTO e     ON e.SK_ESTABELECIMENTO = f.SK_ESTABELECIMENTO
    JOIN DIM_CARATER_ATENDIMENTO c ON c.SK_CARATER = f.SK_CARATER
    JOIN DIM_CID d                 ON d.SK_CID = f.SK_CID
    WHERE f.SK_MUNICIPIO_RESIDENCIA <> -1 AND f.SK_ESTABELECIMENTO <> -1
    GROUP BY m.COD_IBGE_6, e.COD_CNES
    HAVING SUM(f.QTD_INTERNACAO) > 0
""")
gravar_grafo(driver, """
    UNWIND $linhas AS r
    MATCH (m:Municipio {cod_ibge: r.mun}), (h:Hospital {cnes: r.cnes})
    MERGE (m)-[i:INTERNOU_EM]->(h)
    SET i.internacoes = r.internacoes, i.valor = r.valor, i.dias = r.dias,
        i.urgencias = r.urgencias, i.respiratorias = r.respiratorias
""", df, "INTERNOU_EM")

## Especialidades: atendidas pelo hospital e demandadas pelo município

In [ ]:
df = consultar("""
    SELECT e.COD_CNES AS CNES, s.COD_ESPECIALIDADE AS COD, s.NOME_ESPECIALIDADE AS NOME,
           SUM(f.QTD_INTERNACAO) AS INTERNACOES, SUM(f.QTD_DIAS_PERMANENCIA) AS DIAS
    FROM FATO_INTERNACAO f
    JOIN DIM_ESTABELECIMENTO e ON e.SK_ESTABELECIMENTO = f.SK_ESTABELECIMENTO
    JOIN DIM_ESPECIALIDADE s   ON s.SK_ESPECIALIDADE = f.SK_ESPECIALIDADE
    WHERE f.SK_ESTABELECIMENTO <> -1 AND f.SK_ESPECIALIDADE <> -1
    GROUP BY e.COD_CNES, s.COD_ESPECIALIDADE, s.NOME_ESPECIALIDADE
    HAVING SUM(f.QTD_INTERNACAO) > 0
""")
gravar_grafo(driver, """
    UNWIND $linhas AS r
    MERGE (s:Especialidade {cod: r.cod}) SET s.nome = r.nome
    WITH s, r
    MATCH (h:Hospital {cnes: r.cnes})
    MERGE (h)-[a:ATENDEU]->(s)
    SET a.internacoes = r.internacoes, a.dias = r.dias
""", df, "ATENDEU")

df = consultar("""
    SELECT m.COD_IBGE_6 AS MUN, s.COD_ESPECIALIDADE AS COD, SUM(f.QTD_INTERNACAO) AS INTERNACOES
    FROM FATO_INTERNACAO f
    JOIN DIM_MUNICIPIO m     ON m.SK_MUNICIPIO = f.SK_MUNICIPIO_RESIDENCIA
    JOIN DIM_ESPECIALIDADE s ON s.SK_ESPECIALIDADE = f.SK_ESPECIALIDADE
    WHERE f.SK_MUNICIPIO_RESIDENCIA <> -1 AND f.SK_ESPECIALIDADE <> -1
    GROUP BY m.COD_IBGE_6, s.COD_ESPECIALIDADE
    HAVING SUM(f.QTD_INTERNACAO) > 0
""")
gravar_grafo(driver, """
    UNWIND $linhas AS r
    MATCH (m:Municipio {cod_ibge: r.mun}), (s:Especialidade {cod: r.cod})
    MERGE (m)-[d:DEMANDOU]->(s)
    SET d.internacoes = r.internacoes
""", df, "DEMANDOU")

## Procedimentos realizados por hospital, com custo médio

In [ ]:
df = consultar("""
    SELECT COD_PROCEDIMENTO AS COD, NOME_PROCEDIMENTO AS NOME, COD_CNES AS CNES,
           QTD_INTERNACOES AS INTERNACOES, VL_CUSTO_MEDIO AS CUSTO_MEDIO,
           VL_CUSTO_MEDIO_ESTADO AS CUSTO_MEDIO_ESTADO
    FROM MART_P05_CUSTO_PROCEDIMENTO
""")
gravar_grafo(driver, """
    UNWIND $linhas AS r
    MERGE (p:Procedimento {cod: r.cod})
    SET p.nome = r.nome, p.custo_medio_estado = r.custo_medio_estado
    WITH p, r
    MATCH (h:Hospital {cnes: r.cnes})
    MERGE (h)-[x:REALIZOU]->(p)
    SET x.internacoes = r.internacoes, x.custo_medio = r.custo_medio
""", df, "REALIZOU")

## Meses: produção por município, ocupação por hospital e doenças respiratórias

In [ ]:
df = consultar("""
    SELECT ANO_MES, NR_ANO AS ANO, NR_MES AS MES, COD_IBGE_6 AS MUN,
           QTD_AMBULATORIAL, VL_AMBULATORIAL, QTD_INTERNACOES AS INTERNACOES,
           VL_HOSPITALAR, VL_TOTAL
    FROM MART_P01_CUSTO_VOLUME_MUNICIPIO
""")
gravar_grafo(driver, """
    UNWIND $linhas AS r
    MERGE (t:Mes {ano_mes: r.ano_mes}) SET t.ano = r.ano, t.mes = r.mes
    WITH t, r
    MATCH (m:Municipio {cod_ibge: r.mun})
    MERGE (m)-[p:PRODUZIU]->(t)
    SET p.qtd_ambulatorial = r.qtd_ambulatorial, p.vl_ambulatorial = r.vl_ambulatorial,
        p.internacoes = r.internacoes, p.vl_hospitalar = r.vl_hospitalar, p.vl_total = r.vl_total
""", df, "PRODUZIU")

df = consultar("""
    SELECT ANO_MES, COD_CNES AS CNES, QTD_LEITOS_SUS AS LEITOS_SUS,
           QTD_PACIENTES_DIA AS PACIENTES_DIA, QTD_LEITOS_DIA AS LEITOS_DIA, PCT_OCUPACAO AS PCT
    FROM MART_P04_OCUPACAO_HOSPITALAR
""")
gravar_grafo(driver, """
    UNWIND $linhas AS r
    MATCH (h:Hospital {cnes: r.cnes}), (t:Mes {ano_mes: r.ano_mes})
    MERGE (h)-[o:OCUPACAO]->(t)
    SET o.leitos_sus = r.leitos_sus, o.pacientes_dia = r.pacientes_dia,
        o.leitos_dia = r.leitos_dia, o.pct = r.pct
""", df, "OCUPACAO")

df = consultar("""
    SELECT ANO_MES, NOME_MES, QTD_INTERNACOES AS INTERNACOES, VL_TOTAL AS VALOR,
           NR_RANKING_ANO AS RANKING_ANO
    FROM MART_P07_RESPIRATORIAS_MENSAL
""")
gravar_grafo(driver, """
    UNWIND $linhas AS r
    MERGE (g:GrupoDoenca {nome: 'DOENCAS RESPIRATORIAS (CID J00-J99)'})
    WITH g, r
    MATCH (t:Mes {ano_mes: r.ano_mes})
    SET t.nome = r.nome_mes
    MERGE (g)-[x:INTERNACOES_NO_MES]->(t)
    SET x.internacoes = r.internacoes, x.valor = r.valor, x.ranking_ano = r.ranking_ano
""", df, "INTERNACOES_NO_MES")

## Perfil do paciente nas internações de urgência

In [ ]:
df = consultar("""
    SELECT p.SEXO || '|' || p.FAIXA_ETARIA AS CHAVE, p.NOME_SEXO AS SEXO, p.FAIXA_ETARIA AS FAIXA,
           e.COD_CNES AS CNES, SUM(f.QTD_INTERNACAO) AS INTERNACOES
    FROM FATO_INTERNACAO f
    JOIN DIM_PERFIL_PACIENTE p     ON p.SK_PERFIL = f.SK_PERFIL
    JOIN DIM_CARATER_ATENDIMENTO c ON c.SK_CARATER = f.SK_CARATER
    JOIN DIM_ESTABELECIMENTO e     ON e.SK_ESTABELECIMENTO = f.SK_ESTABELECIMENTO
    WHERE c.TIPO_CARATER = 'URGENCIA' AND f.SK_ESTABELECIMENTO <> -1 AND f.SK_PERFIL <> -1
    GROUP BY p.SEXO, p.NOME_SEXO, p.FAIXA_ETARIA, e.COD_CNES
    HAVING SUM(f.QTD_INTERNACAO) > 0
""")
gravar_grafo(driver, """
    UNWIND $linhas AS r
    MERGE (p:Perfil {chave: r.chave}) SET p.sexo = r.sexo, p.faixa_etaria = r.faixa
    WITH p, r
    MATCH (h:Hospital {cnes: r.cnes})
    MERGE (p)-[u:INTERNADO_EM_URGENCIA]->(h)
    SET u.internacoes = r.internacoes
""", df, "INTERNADO_EM_URGENCIA")

## Transbordo entre regiões de saúde

Relação derivada do próprio grafo: soma as internações de moradores de uma região
atendidos em hospitais de outra região.

In [ ]:
with driver.session() as s:
    r = s.run("""
        MATCH (ro:Regiao)<-[:PERTENCE_A]-(:Municipio)-[i:INTERNOU_EM]->(:Hospital)
              -[:LOCALIZADO_EM]->(:Municipio)-[:PERTENCE_A]->(rd:Regiao)
        WHERE ro <> rd AND ro.nome <> 'FORA DE ALAGOAS'
        WITH ro, rd, sum(i.internacoes) AS internacoes
        MERGE (ro)-[t:TRANSBORDO]->(rd)
        SET t.internacoes = internacoes
        RETURN count(t) AS qtd
    """).single()
print(f"TRANSBORDO: {r['qtd']} relações derivadas")

## Resumo do grafo

In [ ]:
with driver.session() as s:
    nos = s.run("MATCH (n) RETURN labels(n)[0] AS rotulo, count(*) AS qtd ORDER BY rotulo").data()
    rels = s.run("MATCH ()-[r]->() RETURN type(r) AS tipo, count(*) AS qtd ORDER BY tipo").data()
driver.close()
for x in nos:
    print(f"{x['rotulo']}: {x['qtd']} nós")
for x in rels:
    print(f"{x['tipo']}: {x['qtd']} relações")
print(f"grafo: {sum(x['qtd'] for x in nos)} nós, {sum(x['qtd'] for x in rels)} relações")